# Get raw data

In [1]:
import os
import time
import requests

In [4]:
RE_TRIAL = 3
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

def download(start_year: int, end_year: int, data_dir: str):
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            month_label = f"{year}-{month:02d}"
            filename = f"yellow_tripdata_{month_label}.parquet"
            out_path = os.path.join(data_dir, filename)

            if os.path.exists(out_path):
                print(f"\n[SKIP] {month_label} already exists\n")
                continue

            url = f"{BASE_URL}/{filename}"

            for i in range(RE_TRIAL):
                print(f"\n[DOWNLOAD] {month_label} ...")
                try:
                    resp = requests.get(url, stream=True, timeout=30)
                    resp.raise_for_status()
                    with open(out_path, "wb") as f:
                        for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                            f.write(chunk)
                    print(f"  -> saved to {out_path}\n")
                    break
                except Exception as e:
                    print(f"Error at {month_label} - {e}")
                    if os.path.exists(out_path):
                        os.remove(out_path)
                    time.sleep(10)
                    print(f"\nRetry to download {month_label} #{i+1}")
            else:
                print(f"\n[FAIL] {month_label} could not be downloaded\n")

            break

    print("Done!")

In [ ]:
data_dir = "../../data"
os.makedirs(data_dir, exist_ok=True)

download(start_year=2020, end_year=2020, data_dir=data_dir)


[DOWNLOAD] 2020-01 ...
  -> saved to ./data/yellow_tripdata_2020-01.parquet

Done!


In [16]:
import pandas as pd

df = pd.read_parquet('../../data/yellow_tripdata_2018-01.parquet')
print(df.shape)
df.info()

(8760123, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760123 entries, 0 to 8760122
Data columns (total 21 columns):
 #   Column                Dtype         
---  ------                -----         
 0   vendorID              object        
 1   tpepPickupDateTime    datetime64[ns]
 2   tpepDropoffDateTime   datetime64[ns]
 3   passengerCount        int32         
 4   tripDistance          float64       
 5   puLocationId          object        
 6   doLocationId          object        
 7   startLon              float64       
 8   startLat              float64       
 9   endLon                float64       
 10  endLat                float64       
 11  rateCodeId            int32         
 12  storeAndFwdFlag       object        
 13  paymentType           object        
 14  fareAmount            float64       
 15  extra                 float64       
 16  mtaTax                float64       
 17  improvementSurcharge  object        
 18  tipAmount             float6

In [17]:
df = pd.read_parquet('../../data/yellow_tripdata_2019-06.parquet')
print(df.shape)
df.info()

(6940643, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6940643 entries, 0 to 6940642
Data columns (total 21 columns):
 #   Column                Dtype         
---  ------                -----         
 0   vendorID              object        
 1   tpepPickupDateTime    datetime64[ns]
 2   tpepDropoffDateTime   datetime64[ns]
 3   passengerCount        int32         
 4   tripDistance          float64       
 5   puLocationId          object        
 6   doLocationId          object        
 7   startLon              float64       
 8   startLat              float64       
 9   endLon                float64       
 10  endLat                float64       
 11  rateCodeId            int32         
 12  storeAndFwdFlag       object        
 13  paymentType           object        
 14  fareAmount            float64       
 15  extra                 float64       
 16  mtaTax                float64       
 17  improvementSurcharge  object        
 18  tipAmount             float6

In [18]:
df = pd.read_parquet('../../data/yellow_tripdata_2020-01.parquet')
print(df.shape)
df.info()

(1, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   vendorID              1 non-null      object        
 1   tpepPickupDateTime    1 non-null      datetime64[ns]
 2   tpepDropoffDateTime   1 non-null      datetime64[ns]
 3   passengerCount        1 non-null      int32         
 4   tripDistance          1 non-null      float64       
 5   puLocationId          1 non-null      object        
 6   doLocationId          1 non-null      object        
 7   startLon              0 non-null      float64       
 8   startLat              0 non-null      float64       
 9   endLon                0 non-null      float64       
 10  endLat                0 non-null      float64       
 11  rateCodeId            1 non-null      int32         
 12  storeAndFwdFlag       1 non-null      object        
 13  paymentType     

Chuyển hết lên `S3`

In [ ]:
from pathlib import Path

parquet_files = list(Path('../../data').rglob('*.parquet'))
for filepath in parquet_files:
    !aws s3 cp {filepath} s3://flezi-agent/nyc-taxi/{filepath.name}
    # break

upload: ../../data/yellow_tripdata_2010-10.parquet to s3://flezi-agent/nyc-taxi/yellow_tripdata_2010-10.parquet
upload: ../../data/yellow_tripdata_2010-05.parquet to s3://flezi-agent/nyc-taxi/yellow_tripdata_2010-05.parquet
upload: ../../data/yellow_tripdata_2009-06.parquet to s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-06.parquet
upload: ../../data/yellow_tripdata_2009-04.parquet to s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-04.parquet
upload: ../../data/yellow_tripdata_2010-06.parquet to s3://flezi-agent/nyc-taxi/yellow_tripdata_2010-06.parquet
upload: ../../data/yellow_tripdata_2010-03.parquet to s3://flezi-agent/nyc-taxi/yellow_tripdata_2010-03.parquet
upload: ../../data/yellow_tripdata_2009-03.parquet to s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-03.parquet
upload: ../../data/yellow_tripdata_2009-02.parquet to s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-02.parquet
upload: ../../data/yellow_tripdata_2009-12.parquet to s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-12.